In [16]:
import re
import sqlite3
import logging
import pandas as pd
from pathlib import Path

In [17]:
DB_PATH = "railway.db"
BLOCK_TRIGGER = "EXCEPTION REPORT"
HEADER_SCAN_ROWS = 10
LOG_FORMAT = "%(asctime)s [%(levelname)s] %(message)s"
 
logging.basicConfig(level=logging.INFO, format=LOG_FORMAT)
log = logging.getLogger(__name__)

In [18]:
def detect_blocks(df: pd.DataFrame) -> list:
    """
    Scan sheet for rows containing BLOCK_TRIGGER keyword.
    Return list of sub-DataFrames, one per block.
    Only blocks that start with EXCEPTION REPORT are included —
    any content above the first trigger is completely ignored.
    """
    trigger_rows = []
    for idx, row in df.iterrows():
        row_text = " ".join(str(v) for v in row.values if pd.notna(v))
        if BLOCK_TRIGGER.lower() in row_text.lower():
            trigger_rows.append(idx)
 
    if not trigger_rows:
        log.debug("  No blocks found in this sheet.")
        return []
 
    blocks = []
    for i, start in enumerate(trigger_rows):
        # Each block ends where the next one begins (or at the sheet end)
        end = trigger_rows[i + 1] if i + 1 < len(trigger_rows) else df.index[-1] + 1
        block = df.loc[start:end - 1].reset_index(drop=True)
        blocks.append(block)
 
    log.info(f"  Detected {len(blocks)} block(s).")
    return blocks

In [19]:
def extract_metadata(block: pd.DataFrame) -> dict:
    """
    Flatten first HEADER_SCAN_ROWS rows into one string.
    Extract key metadata fields via regex.
    """
    header_rows = block.iloc[:HEADER_SCAN_ROWS]
    cells = []
    for _, row in header_rows.iterrows():
        for val in row.values:
            if pd.notna(val) and str(val).strip():
                cells.append(str(val).strip())
 
    full_text = " | ".join(cells)
 
    def search(pattern, text=full_text, group=1, default=""):
        m = re.search(pattern, text, re.IGNORECASE)
        return m.group(group).strip() if m else default
 
    metadata = {
        "full_header_text": full_text,
        "section":   search(r"section[:\s#-]*([A-Z0-9/ ]+?)(?:\||$|km|trc|run)", full_text),
        "trc_no":    search(r"trc[\s#no.:]*([A-Z0-9\-/]+)", full_text),
        "run_date":  search(r"(?:run\s*date|date)[:\s]*(\d{1,2}[\/\-\.]\d{1,2}[\/\-\.]\d{2,4})", full_text),
        "run_no":    search(r"run[\s#no.:]*(\d+)", full_text),
        "defect":    search(r"defect[:\s]*([A-Za-z0-9 _\-]+?)(?:\||$|rail|section)", full_text),
        "rail_side": search(r"\b(left|right|lh|rh)\b", full_text),
        "km_range":  search(r"km[:\s]*([\d.]+\s*[-–to]+\s*[\d.]+)", full_text),
    }
 
    log.debug(f"  Metadata: section={metadata['section']!r}, trc={metadata['trc_no']!r}")
    return metadata

In [20]:
SERIAL_NO_PATTERN = re.compile(
    r"^\s*(?:s\.?\s*(?:r|l)?\.?\s*no\.?|s\s*no\.?|sno\.?|sl\.?\s*no\.?|sr\.?\s*no\.?|no\.?)\s*$",
    re.IGNORECASE
)

In [21]:
def _is_serial_no_row(row: pd.Series) -> bool:
    """Return True if any cell in the row matches a serial-number header pattern."""
    for val in row.values:
        if pd.notna(val) and SERIAL_NO_PATTERN.match(str(val).strip()):
            return True
    return False

In [22]:
def _is_numeric(val) -> bool:
    """Return True if val can be converted to a float (i.e. a real data row)."""
    try:
        float(str(val).strip())
        return True
    except (ValueError, TypeError):
        return False

In [23]:
def extract_table(block: pd.DataFrame):
    """
    Locate the S.No header row using the broadened pattern.
    Extract rows below it as a structured DataFrame.
    FIX 4: Drop footer/summary rows (e.g. 'Total length having wear...')
            by keeping only rows where the first column is numeric.
    Returns None if no valid table is found.
    """
    header_row_idx = None
    for i, row in block.iterrows():
        if _is_serial_no_row(row):
            header_row_idx = i
            break
 
    if header_row_idx is None:
        log.warning("  Serial-number header row not found — skipping table extraction.")
        return None
 
    header = block.loc[header_row_idx].tolist()
    data_rows = block.loc[header_row_idx + 1:].reset_index(drop=True)
 
    if data_rows.empty:
        log.warning("  No data rows found below header.")
        return None
 
    # Assign column names from the header row
    data_rows.columns = [
        str(h).strip() if pd.notna(h) and str(h).strip() else f"col_{i}"
        for i, h in enumerate(header)
    ]
 
    # Handle optional sub-header row (e.g. "Km" / "Meter" under "Start Location")
    first_data_row = data_rows.iloc[0]
    if all(
        str(v).lower() in ("km", "meter", "m", "nan", "")
        for v in first_data_row.values
        if pd.notna(v)
    ):
        combined = []
        for col, sub in zip(data_rows.columns, first_data_row.values):
            if pd.notna(sub) and str(sub).strip().lower() not in ("nan", ""):
                combined.append(f"{col}_{sub}")
            else:
                combined.append(col)
        data_rows.columns = combined
        data_rows = data_rows.iloc[1:].reset_index(drop=True)
 
    # FIX 4: Drop footer / summary / total rows by checking the first column is numeric
    # This removes rows like "Total length having wear more than threshold"
    first_col = data_rows.columns[0]
    data_rows = data_rows[data_rows[first_col].apply(_is_numeric)].reset_index(drop=True)
 
    return data_rows

In [24]:
def clean_table(df: pd.DataFrame) -> pd.DataFrame:
    """
    - Drop fully empty rows and auto-generated columns
    - Standardise column names to snake_case
    - FIX 2: Deduplicate column names (s_no → s_no, s_no_1, s_no_2 ...)
    - FIX 4: Second-pass filter — drop any remaining non-numeric s_no rows
    """
    # Drop rows that are entirely NaN
    df = df.dropna(how="all").reset_index(drop=True)
 
    # Drop unnamed / auto-generated columns
    df = df.loc[:, ~df.columns.str.contains(r"^Unnamed|^col_\d+$", na=False)]
 
    # Standardise column names to snake_case
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(r"[^\w]", "_", regex=True)
        .str.replace(r"_+", "_", regex=True)
        .str.strip("_")
    )
 
    # Drop columns whose name reduced to empty string
    df = df.loc[:, df.columns != ""]
 
    # FIX 2: Deduplicate column names by appending _1, _2 etc.
    seen = {}
    new_cols = []
    for col in df.columns:
        if col in seen:
            seen[col] += 1
            new_cols.append(f"{col}_{seen[col]}")
        else:
            seen[col] = 0
            new_cols.append(col)
    df.columns = new_cols
 
    # FIX 4 (second pass): If s_no column exists, keep only numeric rows
    # Catches any footer rows that survived the first pass in extract_table
    sno_col = next((c for c in df.columns if re.match(r"^s_?no", c)), None)
    if sno_col:
        df = df[df[sno_col].apply(_is_numeric)].reset_index(drop=True)
 
    return df

In [25]:
DATASET_TYPE_MAP = {
    "lip flow":       "lip_flow_data",
    "lip_flow":       "lip_flow_data",
    "vegetation":     "vegetation_data",
    "ballast":        "ballast_data",
    "rail defect":    "rail_defects_data",
    "rail_defect":    "rail_defects_data",
    "vertical wear":  "vertical_wear_data",
    "vertical_wear":  "vertical_wear_data",
    "lateral wear":   "lateral_wear_data",
    "lateral rail":   "lateral_wear_data",
    "lateral_wear":   "lateral_wear_data",
    "sleeper":        "sleeper_defects_data",
    "fitting":        "fittings_data",
    "sod":            "sod_data",
    "geometry":       "geometry_data",
    "gauge":          "gauge_data",
    "squat":          "squat_data",
    "corrugation":    "corrugation_data",
    "head check":     "head_check_data",
}

In [26]:
def detect_dataset_type(metadata: dict, file_name: str = "") -> str:
    """
    Determine target SQLite table name from header text, defect field, or filename.
    Falls back to 'general_data'.
    """
    text = (
        metadata.get("full_header_text", "") + " " +
        metadata.get("defect", "") + " " +
        file_name
    ).lower()
 
    for keyword, table_name in DATASET_TYPE_MAP.items():
        if keyword in text:
            log.debug(f"  Dataset type → {table_name}")
            return table_name
 
    log.debug("  Dataset type → general_data (fallback)")
    return "general_data"

In [27]:
def _get_existing_columns(conn: sqlite3.Connection, table_name: str) -> list:
    """Return list of column names for an existing table, or [] if not found."""
    cur = conn.execute(
        "SELECT name FROM sqlite_master WHERE type='table' AND name=?", (table_name,)
    )
    if cur.fetchone() is None:
        return []
    cur = conn.execute(f"PRAGMA table_info('{table_name}')")
    return [row[1] for row in cur.fetchall()]

In [28]:
def _add_missing_columns(conn: sqlite3.Connection, table_name: str, new_cols: list):
    """ALTER TABLE to add columns that are in the DataFrame but not yet in the table."""
    existing = _get_existing_columns(conn, table_name)
    for col in new_cols:
        if col not in existing:
            try:
                conn.execute(f"ALTER TABLE '{table_name}' ADD COLUMN '{col}' TEXT")
                log.debug(f"  Added missing column '{col}' to '{table_name}'")
            except Exception as e:
                log.warning(f"  Could not add column '{col}': {e}")

In [29]:
def store_to_sql(df: pd.DataFrame, table_name: str, db_path: str = DB_PATH):
    """
    FIX 3: Schema-flexible SQLite append.
    - Table doesn't exist  → create it fresh via to_sql.
    - Table already exists → ALTER to add any new columns, then append
                             only the columns common to both.
    """
    try:
        with sqlite3.connect(db_path) as conn:
            existing_cols = _get_existing_columns(conn, table_name)
 
            if not existing_cols:
                # First time writing this table
                df.to_sql(table_name, conn, if_exists="append", index=False)
            else:
                # Align schemas: add any new columns to the existing table
                _add_missing_columns(conn, table_name, df.columns.tolist())
 
                # Re-fetch after ALTER
                existing_cols = _get_existing_columns(conn, table_name)
 
                # Write only columns present in both the DataFrame and the table
                common_cols = [c for c in df.columns if c in existing_cols]
                if not common_cols:
                    log.warning(f"  No common columns with '{table_name}' — skipping.")
                    return
 
                df[common_cols].to_sql(table_name, conn, if_exists="append", index=False)
 
        log.info(f"  Stored {len(df)} row(s) → table '{table_name}' in '{db_path}'")
 
    except Exception as e:
        log.error(f"  DB write error for table '{table_name}': {e}")

In [30]:
def process_folder(folder_path: str, db_path: str = DB_PATH):
    """
    Walk every .xlsx file in folder_path.
    For each file → each sheet → each EXCEPTION REPORT block:
      1. Extract metadata from the header rows
      2. Extract and clean the data table
      3. Drop footer/summary rows (non-numeric S.No)
      4. Attach metadata columns to every row
      5. Detect dataset type → choose SQLite table name
      6. Append rows to SQLite (schema-flexible)
    """
    folder = Path(folder_path)
    xlsx_files = list(folder.glob("*.xlsx"))
 
    if not xlsx_files:
        log.warning(f"No .xlsx files found in: {folder_path}")
        return
 
    log.info(f"Found {len(xlsx_files)} Excel file(s) in '{folder_path}'.")
 
    for file_path in xlsx_files:
        log.info(f"\n{'='*60}")
        log.info(f"Processing file: {file_path.name}")
 
        try:
            xl = pd.ExcelFile(file_path, engine="openpyxl")
        except Exception as e:
            log.error(f"Cannot open '{file_path.name}': {e}")
            continue
 
        for sheet_name in xl.sheet_names:
            log.info(f"  Sheet: '{sheet_name}'")
 
            try:
                # Read entire sheet as raw strings — no header assumption
                raw_df = xl.parse(sheet_name, header=None, dtype=str)
            except Exception as e:
                log.error(f"  Cannot read sheet '{sheet_name}': {e}")
                continue
 
            if raw_df.empty:
                log.info("  Sheet is empty — skipping.")
                continue
 
            # Split sheet into blocks, starting ONLY from EXCEPTION REPORT rows
            # Anything above the first trigger (summary sections etc.) is ignored
            blocks = detect_blocks(raw_df)
 
            for block_idx, block in enumerate(blocks):
                log.info(f"  Processing block {block_idx + 1} ...")
                try:
                    # Step A: parse metadata from the block header area
                    metadata = extract_metadata(block)
 
                    # Step B: locate and extract the data table
                    table_df = extract_table(block)
                    if table_df is None or table_df.empty:
                        log.warning(f"  Block {block_idx + 1}: no usable table — skipping.")
                        continue
 
                    # Step C: clean column names and remove junk rows/cols
                    table_df = clean_table(table_df)
                    if table_df.empty:
                        log.warning(f"  Block {block_idx + 1}: empty after cleaning — skipping.")
                        continue
 
                    # Step D: attach metadata as extra columns on every row
                    table_df["section"]     = metadata.get("section", "")
                    table_df["trc_no"]      = metadata.get("trc_no", "")
                    table_df["run_date"]    = metadata.get("run_date", "")
                    table_df["run_no"]      = metadata.get("run_no", "")
                    table_df["defect"]      = metadata.get("defect", "")
                    table_df["rail_side"]   = metadata.get("rail_side", "")
                    table_df["km_range"]    = metadata.get("km_range", "")
                    table_df["source_file"] = file_path.name
                    table_df["sheet_name"]  = sheet_name
 
                    # Step E: decide which SQLite table to write into
                    table_name = detect_dataset_type(metadata, file_path.name)
 
                    # Step F: persist to database
                    store_to_sql(table_df, table_name, db_path)
 
                except Exception as e:
                    log.error(f"  Block {block_idx + 1} failed: {e}", exc_info=True)
 
    log.info(f"\n{'='*60}")
    log.info(f"Pipeline complete. Database saved to: {db_path}")

In [31]:
# AFTER (hardcoded path)
if __name__ == "__main__":

    input_folder = r"D:\\ENGINEER\\IndianRailwaysProject\\data"   # ← paste your folder path here
    output_db    = "railway.db"                         # ← DB will be created here

    process_folder(input_folder, output_db)

2026-04-08 18:32:06,563 [INFO] Found 8 Excel file(s) in 'D:\\ENGINEER\\IndianRailwaysProject\\data'.
2026-04-08 18:32:06,564 [INFO] 
2026-04-08 18:32:06,567 [INFO] Processing file: 1 SOD exception.xlsx
2026-04-08 18:32:07,657 [INFO]   Sheet: 'KYN-MMR UP'
2026-04-08 18:32:07,665 [INFO]   Sheet: 'KYN-MMR DN'
2026-04-08 18:32:07,672 [INFO]   Detected 1 block(s).
2026-04-08 18:32:07,673 [INFO]   Processing block 1 ...
2026-04-08 18:32:07,712 [INFO]   Stored 1 row(s) → table 'sod_data' in 'railway.db'
2026-04-08 18:32:07,713 [INFO]   Sheet: 'MMR-CSN UP'
2026-04-08 18:32:07,721 [INFO]   Detected 1 block(s).
2026-04-08 18:32:07,723 [INFO]   Processing block 1 ...
2026-04-08 18:32:07,749 [INFO]   Stored 1 row(s) → table 'sod_data' in 'railway.db'
2026-04-08 18:32:07,751 [INFO]   Sheet: 'MMR-CSN DN'
2026-04-08 18:32:07,756 [INFO]   Detected 1 block(s).
2026-04-08 18:32:07,758 [INFO]   Processing block 1 ...
2026-04-08 18:32:07,783 [INFO]   Stored 1 row(s) → table 'sod_data' in 'railway.db'
2026